# Task 3 — Define Input Schema and Validate

This notebook defines the common input schema and validates the cleaned PNU dataset against the required rules.

The validation checks include:
- Required columns
- Data structure
- Allowed values
- Duplicate handling
- Required field validation

Valid records will be saved as validated.csv, while failed records will be saved as rejected.csv with validation reasons.

In [151]:
# Import required libraries for data validation

import pandas as pd
from pathlib import Path

In [152]:
# Define project directories

project_root = Path("..")

interim_dir = project_root / "data" / "interim"

In [153]:
# Load cleaned PNU dataset generated from Task 2

input_path = interim_dir / "PNU_cleaned.csv"

df = pd.read_csv(input_path)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Dataset loaded successfully
Rows: 10473
Columns: 13


,research_id,university,title,authors,publication_year,publication_date,abstract,research_field,tech_category,journal,doi,url,source
0,PNU-0,PNU,sexual workplace violence in the health sector...,NaN,2023,NaN,NaN,humanitarian,NaN,computer systems science and engineering,NaN,NaN,PNU Open Data
1,PNU-1,PNU,spatial analysis of accessibility to urban gre...,NaN,2023,NaN,NaN,humanitarian,NaN,intelligent automation and soft computing,NaN,NaN,PNU Open Data
2,PNU-2,PNU,"association of ccnd1 (c.723g > a, rs9344) vari...",NaN,2023,NaN,NaN,humanitarian,NaN,intelligent automation and soft computing,NaN,NaN,PNU Open Data
3,PNU-3,PNU,neuroprotective effect of barbaloin on strepto...,NaN,2023,NaN,NaN,humanitarian,NaN,computer systems science and engineering,NaN,NaN,PNU Open Data
4,PNU-4,PNU,adoption of antenatal care conversation mappin...,NaN,2023,NaN,NaN,humanitarian,NaN,peerj computer science,NaN,NaN,PNU Open Data


In [154]:
# Define the common schema required for all university datasets

schema = pd.DataFrame([
    ["research_id", "string", False, "Non-empty and unique"],
    ["university", "string", False, "PNU"],
    ["title", "string", False, "Non-empty text"],
    ["authors", "string", False, "Text value"],
    ["publication_year", "integer", False, "Valid publication year"],
    ["publication_date", "date", False, "Valid date format"],
    ["abstract", "string", False, "Text value"],
    ["research_field", "string", False, "Text value"],
    ["tech_category", "string", False, "Text value"],
    ["journal", "string", False, "Text value"],
    ["doi", "string", False, "DOI value"],
    ["url", "string", False, "Valid URL"],
    ["source", "string", False, "Dataset source"]
],
columns=[
    "column",
    "expected_type",
    "nullable",
    "allowed_values_or_range"
])

schema

,column,expected_type,nullable,allowed_values_or_range
0,research_id,string,False,Non-empty and unique
1,university,string,False,PNU
2,title,string,False,Non-empty text
3,authors,string,False,Text value
4,publication_year,integer,False,Valid publication year
5,publication_date,date,False,Valid date format
6,abstract,string,False,Text value
7,research_field,string,False,Text value
8,tech_category,string,False,Text value
9,journal,string,False,Text value


In [155]:
# Check if dataset columns match the common schema

expected_columns = schema["column"].tolist()

print("PNU rows:", len(df))
print("PNU columns:", len(df.columns))

print(
    "Columns match schema:",
    df.columns.tolist() == expected_columns
)

print(
    "Duplicate research IDs:",
    df["research_id"].duplicated().sum()
)

PNU rows: 10473
PNU columns: 13
Columns match schema: True
Duplicate research IDs: 0


In [156]:
# Function to validate dataset against schema rules

def validate_dataset(df):

    result = df.copy()

    # Create validation error column
    errors = pd.Series(
        "",
        index=result.index,
        dtype="string"
    )


    # Check missing values in required columns
    required_columns = schema[
        schema["nullable"] == False
    ]["column"].tolist()


    for column in required_columns:

        missing = (
            result[column].isna()
            |
            result[column].astype("string").str.strip().eq("")
        )

        errors.loc[missing] += f"{column} is missing; "


    # Check duplicate research IDs

    duplicate_ids = result["research_id"].duplicated(
        keep=False
    )

    errors.loc[duplicate_ids] += "duplicate research_id; "


    # Check publication year format

    invalid_year = pd.to_numeric(
        result["publication_year"],
        errors="coerce"
    ).isna()

    errors.loc[invalid_year] += "invalid publication year; "


    # Check URL format

    invalid_url = ~result["url"].astype("string").str.match(
        r"^https?://",
        na=False
    )

    errors.loc[invalid_url] += "invalid URL; "


    # Remove extra semicolon

    result["validation_error"] = (
        errors.str.rstrip("; ")
    )


    # Split valid and rejected records

    validated = result[
        result["validation_error"] == ""
    ].copy()


    rejected = result[
        result["validation_error"] != ""
    ].copy()


    return validated, rejected

In [157]:
# Apply validation rules to PNU dataset

pnu_validated, pnu_rejected = validate_dataset(df)


print("PNU total:", len(df))
print("PNU validated:", len(pnu_validated))
print("PNU rejected:", len(pnu_rejected))

PNU total: 10473
PNU validated: 0
PNU rejected: 10473


In [158]:
# Display validation failure reasons if any records fail

pnu_rejected["validation_error"].value_counts()

validation_error
authors is missing; publication_date is missing; abstract is missing; tech_category is missing; doi is missing; url is missing; invalid URL    10473
Name: count, dtype: Int64

In [159]:
# Save validated and rejected records for the next pipeline stage

pnu_validated.to_csv(
    interim_dir / "validated.csv",
    index=False
)

pnu_rejected.to_csv(
    interim_dir / "rejected.csv",
    index=False
)


print("Validation files saved successfully.")

Validation files saved successfully.
